In [1]:
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from chromadb import PersistentClient
from tqdm import tqdm
import numpy as np
from sklearn.manifold import TSNE
import plotly.graph_objects as go
import os

In [2]:
load_dotenv(override=True)
AZURE_ENDPOINT = (
    "https://ankitsinghtheweeknd691-9348-reso.services.ai.azure.com/openai/v1"
)


collection_name = "docs"
DEFAULT_MODEL = "mistral-medium-latest"
embedding_model = "qwen--qwen3-embedding-8b"
embedding_endpoint = "https://ankitsinghtheweeknd691-9348-reso.services.ai.azure.com/openai/v1/chat/completions"

DB_NAME = "preprocessed_db"
KNOWLEDGE_BASE_PATH = Path("../knowledge_base")
AVERAGE_CHUNK_SIZE = 500
CACHE_DIR = Path("chunk_cache")
CACHE_DIR.mkdir(exist_ok=True)


client = OpenAI(
    api_key=os.getenv("MISTRIAL_API_KEY"),
    base_url="https://api.mistral.ai/v1",
)


In [3]:
def fetch_documents():
    """A homemade version of the LangChain DirectoryLoader"""

    documents = []

    for folder in KNOWLEDGE_BASE_PATH.iterdir():
        doc_type = folder.name
        for file in folder.rglob("*.md"):
            with open(file, "r", encoding="utf-8") as f:
                documents.append(
                    {"type": doc_type, "source": file.as_posix(), "text": f.read()}
                )

    print(f"Loaded {len(documents)} documents")
    return documents


In [4]:
documents = fetch_documents()
print((documents[0]["type"]))
print("\n")
print((documents[0]["text"]))


Loaded 76 documents
company


# About Insurellm

Insurellm was founded by Avery Lancaster in 2015 as an insurance tech startup designed to disrupt an industry in need of innovative products. Its first product was Markellm, the marketplace connecting consumers with insurance providers.

The company experienced rapid growth in its first five years, expanding its product portfolio to include Carllm (auto insurance portal), Homellm (home insurance portal), and Rellm (enterprise reinsurance platform). By 2020, Insurellm had reached a peak of 200 employees with 12 offices across the US.

However, the company underwent a strategic restructuring in 2022-2023 to focus on profitability and sustainable growth. This included consolidating office locations, implementing a remote-first strategy, and streamlining operations. As of 2025, Insurellm operates with a lean, highly efficient team of 32 employees who have built a portfolio of 32 active contracts spanning all eight product lines. The company 

In [5]:
class Result(BaseModel):
    page_content: str
    metadata: dict


In [6]:
# A class to perfectly represent a chunk


class Chunk(BaseModel):
    headline: str = Field(
        description="A brief heading for this chunk, typically a few words, that is most likely to be surfaced in a query"
    )
    summary: str = Field(
        description="A few sentences summarizing the content of this chunk to answer common questions"
    )
    original_text: str = Field(
        description="The original text of this chunk from the provided document, exactly as is, not changed in any way"
    )

    def as_result(self, document):
        metadata = {"source": document["source"], "type": document["type"]}
        return Result(
            page_content=self.headline
            + "\n\n"
            + self.summary
            + "\n\n"
            + self.original_text,
            metadata=metadata,
        )


class Chunks(BaseModel):
    chunks: list[Chunk]


In [7]:
def fetch_documents():
    """A homemade version of the LangChain DirectoryLoader"""

    documents = []

    for folder in KNOWLEDGE_BASE_PATH.iterdir():
        doc_type = folder.name
        for file in folder.rglob("*.md"):
            with open(file, "r", encoding="utf-8") as f:
                documents.append(
                    {"type": doc_type, "source": file.as_posix(), "text": f.read()}
                )

    print(f"Loaded {len(documents)} documents")
    return documents


In [8]:
documents = fetch_documents()


Loaded 76 documents


In [9]:
def make_prompt(document):
    how_many = (len(document["text"]) // AVERAGE_CHUNK_SIZE) + 1
    return f"""
You take a document and you split the document into overlapping chunks for a KnowledgeBase.

The document is from the shared drive of a company called Insurellm.
The document is of type: {document["type"]}
The document has been retrieved from: {document["source"]}

A chatbot will use these chunks to answer questions about the company.
You should divide up the document as you see fit, being sure that the entire document is returned in the chunks - don't leave anything out.
This document should probably be split into {how_many} chunks, but you can have more or less as appropriate.
There should be overlap between the chunks as appropriate; typically about 25% overlap or about 50 words, so you have the same text in multiple chunks for best retrieval results.

For each chunk, you should provide a headline, a summary, and the original text of the chunk.
Together your chunks should represent the entire document with overlap.

Here is the document:

{document["text"]}

Respond with the chunks.
"""


In [10]:
print(make_prompt(documents[0]))



You take a document and you split the document into overlapping chunks for a KnowledgeBase.

The document is from the shared drive of a company called Insurellm.
The document is of type: company
The document has been retrieved from: ../knowledge_base/company/about.md

A chatbot will use these chunks to answer questions about the company.
You should divide up the document as you see fit, being sure that the entire document is returned in the chunks - don't leave anything out.
This document should probably be split into 5 chunks, but you can have more or less as appropriate.
There should be overlap between the chunks as appropriate; typically about 25% overlap or about 50 words, so you have the same text in multiple chunks for best retrieval results.

For each chunk, you should provide a headline, a summary, and the original text of the chunk.
Together your chunks should represent the entire document with overlap.

Here is the document:

# About Insurellm

Insurellm was founded by Avery

In [11]:
def make_messages(document):
    return [
        {"role": "user", "content": make_prompt(document)},
    ]


In [12]:
make_messages(documents[0])


[{'role': 'user',
  'content': "\nYou take a document and you split the document into overlapping chunks for a KnowledgeBase.\n\nThe document is from the shared drive of a company called Insurellm.\nThe document is of type: company\nThe document has been retrieved from: ../knowledge_base/company/about.md\n\nA chatbot will use these chunks to answer questions about the company.\nYou should divide up the document as you see fit, being sure that the entire document is returned in the chunks - don't leave anything out.\nThis document should probably be split into 5 chunks, but you can have more or less as appropriate.\nThere should be overlap between the chunks as appropriate; typically about 25% overlap or about 50 words, so you have the same text in multiple chunks for best retrieval results.\n\nFor each chunk, you should provide a headline, a summary, and the original text of the chunk.\nTogether your chunks should represent the entire document with overlap.\n\nHere is the document:\n\n

In [13]:
def get_cache_path(document):
    return CACHE_DIR / (Path(document["source"]).stem + ".json")


In [ ]:
import time
from openai import RateLimitError, BadRequestError
from tenacity import (
    retry,
    stop_after_attempt,
    wait_exponential_jitter,
    retry_if_exception_type,
)


@retry(
    retry=retry_if_exception_type(RateLimitError),
    wait=wait_exponential_jitter(initial=1, max=30),
    stop=stop_after_attempt(5),
    reraise=True,  # after exhausting retries, raise the original RateLimitError
)
def call_chat_completion(messages):
    return client.chat.completions.create(
        model=DEFAULT_MODEL,
        messages=messages,
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "Chunks",
                "schema": Chunks.model_json_schema(),
                "strict": True,
            },
        },
    )


def process_document(document):
    print(f"Processing: {document['source']}")

    cache_path = get_cache_path(document)

    # Load from cache if available
    if cache_path.exists():
        print(f"Loading cache: {document['source']}")
        parsed = Chunks.model_validate_json(cache_path.read_text(encoding="utf-8"))
        return [chunk.as_result(document) for chunk in parsed.chunks]

    messages = make_messages(document)

    try:
        response = call_chat_completion(messages)
    except BadRequestError as e:
        print(f"\nFAILED: {document['source']}")
        print(e)
        return []
    except RateLimitError as e:
        # tenacity exhausted all 5 attempts
        raise RuntimeError(f"Failed after retries: {document['source']}") from e

    reply = response.choices[0].message.content

    # Parse the response
    parsed = Chunks.model_validate_json(reply)

    # Save to cache
    cache_path.write_text(parsed.model_dump_json(indent=2), encoding="utf-8")

    return [chunk.as_result(document) for chunk in parsed.chunks]


In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed


def create_chunks(documents, max_workers=2):
    chunks = []
    failed = []
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(process_document, doc): doc for doc in documents}
        for future in tqdm(as_completed(futures), total=len(documents)):
            doc = futures[future]
            try:
                chunks.extend(future.result())
            except RuntimeError as e:
                print(f"Skipping {doc['source']}: {e}")
                failed.append(doc["source"])

    if failed:
        print(f"\n{len(failed)} documents failed after retries: {failed}")

    return chunks


chunks = create_chunks(documents)
print(len(chunks))


Processing: ../knowledge_base/company/about.md
Loading cache: ../knowledge_base/company/about.md
Processing: ../knowledge_base/company/careers.md
Loading cache: ../knowledge_base/company/careers.md
Processing: ../knowledge_base/company/culture.md
Processing: ../knowledge_base/company/overview.md
Loading cache: ../knowledge_base/company/culture.md
Loading cache: ../knowledge_base/company/overview.md
Processing: ../knowledge_base/contracts/Contract with Advantage Medical Coverage for Healthllm.md
Loading cache: ../knowledge_base/contracts/Contract with Advantage Medical Coverage for Healthllm.md
Processing: ../knowledge_base/contracts/Contract with Apex Reinsurance for Rellm - AI-Powered Enterprise Reinsurance Solution.md
Processing: ../knowledge_base/contracts/Contract with Atlantic Risk Solutions for Bizllm.md
Loading cache: ../knowledge_base/contracts/Contract with Atlantic Risk Solutions for Bizllm.md
Processing: ../knowledge_base/contracts/Contract with Belvedere Insurance for Marke

 46%|████▌     | 35/76 [00:00<00:00, 348.45it/s]

Processing: ../knowledge_base/contracts/Contract with GlobalRe Partners for Rellm.md
Loading cache: ../knowledge_base/contracts/Contract with GlobalRe Partners for Rellm.md
Processing: ../knowledge_base/contracts/Contract with GreenField Holdings for Markellm.md
Loading cache: ../knowledge_base/contracts/Contract with GreenField Holdings for Markellm.md
Processing: ../knowledge_base/contracts/Contract with Greenstone Insurance for Homellm.md
Loading cache: ../knowledge_base/contracts/Contract with Greenstone Insurance for Homellm.md
Processing: ../knowledge_base/contracts/Contract with GreenValley Insurance for Homellm.md
Loading cache: ../knowledge_base/contracts/Contract with GreenValley Insurance for Homellm.md
Processing: ../knowledge_base/contracts/Contract with Guardian Life Partners for Lifellm.md
Loading cache: ../knowledge_base/contracts/Contract with Guardian Life Partners for Lifellm.md
Processing: ../knowledge_base/contracts/Contract with Harmony Health Plans for Healthllm.

100%|██████████| 76/76 [00:00<00:00, 299.06it/s]

Loading cache: ../knowledge_base/employees/Maxine Thompson.md
Processing: ../knowledge_base/employees/Maya Thompson.md
Loading cache: ../knowledge_base/employees/Maya Thompson.md
Processing: ../knowledge_base/employees/Michael O'Brien.md
Loading cache: ../knowledge_base/employees/Michael O'Brien.md
Processing: ../knowledge_base/employees/Michelle Rivera.md
Loading cache: ../knowledge_base/employees/Michelle Rivera.md
Processing: ../knowledge_base/employees/Nina Patel.md
Processing: ../knowledge_base/employees/Oliver Spencer.md
Loading cache: ../knowledge_base/employees/Oliver Spencer.md
Processing: ../knowledge_base/employees/Priya Sharma.md
Loading cache: ../knowledge_base/employees/Nina Patel.md
Processing: ../knowledge_base/employees/Rachel Martinez.md
Loading cache: ../knowledge_base/employees/Rachel Martinez.md
Loading cache: ../knowledge_base/employees/Priya Sharma.md
Processing: ../knowledge_base/employees/Robert Chen.md
Processing: ../knowledge_base/employees/Samantha Greene.md

In [16]:
print(chunks)

[Result(page_content='Insurellm Overview and Company Background\n\nInsurellm is a remote-first insurance tech firm founded in 2015 with 32 employees and offices in major US cities. It has transitioned from a high-growth startup to a profitable, operationally excellent company.\n\n# Overview of Insurellm\n\nInsurellm is an innovative insurance tech firm with 32 employees operating primarily remotely across the US, with offices in San Francisco (HQ), New York, Austin, Chicago, and Denver.\n\nFounded in 2015, the company has evolved from a high-growth startup to a lean, profitable operation focused on sustainable growth and operational excellence.', metadata={'source': '../knowledge_base/company/overview.md', 'type': 'company'}), Result(page_content='Insurellm Products Overview\n\nInsurellm offers 8 insurance software products, including core insurance portals (Carllm, Homellm, Lifellm, Healthllm, Bizllm) and marketplace & infrastructure solutions (Markellm, Claimllm, Rellm).\n\n## Produc

In [17]:
embedding_client = OpenAI(
    base_url=AZURE_ENDPOINT,
    api_key=os.getenv("AZURE_FOUNDRY_API_KEY"),
)


In [18]:
def create_embeddings(chunks, batch_size=32):
    chroma = PersistentClient(path=DB_NAME)
    if collection_name in [c.name for c in chroma.list_collections()]:
        print("inside here deleted")
        chroma.delete_collection(collection_name)

    texts = [chunk.page_content for chunk in chunks]

    vectors = []
    for i in tqdm(range(0, len(texts), batch_size)):
        batch = texts[i : i + batch_size]
        emb = embedding_client.embeddings.create(
            model=embedding_model, input=batch
        ).data
        vectors.extend([e.embedding for e in emb])

    collection = chroma.get_or_create_collection(collection_name)

    ids = [str(i) for i in range(len(chunks))]
    metas = [chunk.metadata for chunk in chunks]

    collection.add(ids=ids, embeddings=vectors, documents=texts, metadatas=metas)
    print(f"Vectorstore created with {collection.count()} documents")


In [19]:
create_embeddings(chunks)


inside here deleted


100%|██████████| 22/22 [01:07<00:00,  3.05s/it]


Vectorstore created with 697 documents


In [20]:
chroma = PersistentClient(path=DB_NAME)
collection = chroma.get_or_create_collection(collection_name)
result = collection.get(include=["embeddings", "documents", "metadatas"])
vectors = np.array(result["embeddings"])
documents = result["documents"]
metadatas = result["metadatas"]
doc_types = [metadata["type"] for metadata in metadatas]
colors = [
    ["blue", "green", "red", "orange"][
        ["products", "employees", "contracts", "company"].index(t)
    ]
    for t in doc_types
]


In [21]:
tsne = TSNE(n_components=2, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 2D scatter plot
fig = go.Figure(
    data=[
        go.Scatter(
            x=reduced_vectors[:, 0],
            y=reduced_vectors[:, 1],
            mode="markers",
            marker=dict(size=5, color=colors, opacity=0.8),
            text=[
                f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)
            ],
            hoverinfo="text",
        )
    ]
)

fig.update_layout(
    title="2D Chroma Vector Store Visualization",
    scene=dict(xaxis_title="x", yaxis_title="y"),
    width=800,
    height=600,
    margin=dict(r=20, b=10, l=10, t=40),
)

fig.show()


In [22]:
tsne = TSNE(n_components=3, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 3D scatter plot
fig = go.Figure(
    data=[
        go.Scatter3d(
            x=reduced_vectors[:, 0],
            y=reduced_vectors[:, 1],
            z=reduced_vectors[:, 2],
            mode="markers",
            marker=dict(size=5, color=colors, opacity=0.8),
            text=[
                f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)
            ],
            hoverinfo="text",
        )
    ]
)

fig.update_layout(
    title="3D Chroma Vector Store Visualization",
    scene=dict(xaxis_title="x", yaxis_title="y", zaxis_title="z"),
    width=900,
    height=700,
    margin=dict(r=10, b=10, l=10, t=40),
)

fig.show()
